In [1]:
from gensim.models import KeyedVectors
from gensim.scripts.glove2word2vec import glove2word2vec

In [2]:
import numpy as np

In [3]:
def sigmoid(x):
    return 1/(1+np.exp(-x))

In [ ]:
#读取模型
glove_file = "vectors.txt"

由于官方模型无法评估模型（只能类比英文），所以在转换前先自行检查模型是否能使用

In [9]:
#检查模型维度
error_count = 0
error_lines = []

with open(glove_file, "r", encoding="utf-8") as f:
    for line_num, line in enumerate(f, 1):
        parts = line.strip().split()
        if not parts:
            continue
        #维度检查
        first_space = line.find(' ')
        if first_space == -1:
            error_count += 1
            error_lines.append((line_num, "无空格分隔词与向量", line))
            continue
        word_part = line[:first_space]
        vector_part = line[first_space+1:]
        vector_values = vector_part.split()
        if len(vector_values) != 50:
            error_count += 1
            error_lines.append((line_num, word_part, len(vector_values)))
print(f'总错误行数:{error_count}')
print("错误详情（行号|词语|实际维度）")
for line_num, word, actual_dim in error_lines:
    print(f'第{line_num}行 | {word} | {actual_dim}维')

总错误行数:0
错误详情（行号|词语|实际维度）


In [10]:
#加载词向量
word_vectors = {}
with open(glove_file, 'r', encoding='utf-8') as f:
    for line in f:
        word, *vector = line.split()
        word_vectors[word] = np.array(vector, dtype=np.float32)
#检查单词是否在词汇表中
print('雅典' in word_vectors)

True


这里与官方模型相同，测试国家-首都≈ 国家-首都

In [19]:
a='伦敦'
b='英国'
c='北京'
a_vec = word_vectors[a]
b_vec = word_vectors[b]
c_vec = word_vectors[c]
target = b_vec - a_vec + c_vec #关建公式
#找最接近的单词
similarities = {}
for word, vec in word_vectors.items():
    if word not in [a,b,c] and vec.shape== (50,):
        similarities[word] = np.dot(target, vec) / (np.linalg.norm(target) * np.linalg.norm(vec))
end=sorted(similarities.items(), key=lambda x: -x[1])[:5]
print(end)

[('中国', 0.8957616), ('中华人民共和国', 0.80377096), ('此后', 0.7704513), ('上海', 0.76868755), ('先后', 0.7683285)]


伦敦与英国的关系相当于北京与中国的关系，说明模型可行

In [ ]:
w2v_file = "w2v.txt"

In [ ]:
#转换文件，转成word2vec的类型，以使用
glove2word2vec(glove_file, w2v_file)
model = KeyedVectors.load_word2vec_format(w2v_file)

In [14]:
#测试模型
print(model['月亮'])

[-0.823946 -0.450828  0.765004 -0.325522 -0.001223 -0.03465  -0.352132
  0.192825 -0.274168 -0.285092  0.691929  1.019414  0.090842 -0.117143
 -0.131579 -0.110228  0.272736  0.944243 -0.140704  0.175887 -0.634966
 -0.494     0.310001 -0.02545   0.556111  0.396743 -0.790659  1.060223
  0.368573  0.232935  0.584927 -0.917403  0.124081  0.798814  0.035739
 -0.137479  0.312327 -0.984401 -0.284294  0.170804  0.961106 -0.867366
 -0.462011 -0.559952 -0.135531  0.819268  0.888751  0.563657  0.024823
  0.639526]


In [15]:
#测试相似度计算
vector1 = model['月亮']
vector2 = model['月球']

In [16]:
print("余弦距离：‘月亮’与‘月球’的相似度为：{:.3f}".format(vector1.dot(vector2.T)/(np.linalg.norm(vector1) * np.linalg.norm(vector2))))

余弦距离：‘月亮’与‘月球’的相似度为：0.646


In [17]:
for i in range(len(vector1)):
    temp = np.subtract(vector1[i], vector2[i])
    temp = np.power(temp, 2)
    dicts = np.sqrt(temp.sum())
print("欧式距离：‘月亮’与‘月球’的相似度为：{:.3f}".format(dicts))

欧式距离：‘月亮’与‘月球’的相似度为：0.849


转换后可以正常计算两个词的余弦距离与欧式距离